In [1]:
# -*- coding: UTF-8 -*-


#还是要先写好node&graph，定义为QGNode QGGraph，再从QGgraph读信息进networkx，计算自己写
#12.30遇到父节点则把所有子节点视为其一部分，遇到子节点则父节点为其一部分，可否实现？
#结果：父节点的上下游视为所有子节点上下游，反之相同。直接将父子节点传导节点影响力ir能否完成？？？？

In [2]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pylab
import matplotlib as mpl
import pandas as pd
import math

In [3]:
class QGGraphError(Exception):
    #涵盖：输入电流的两节点间不边的异常
    pass


class QGNode():
    def __init__(self, name, decay=0.8, r=0) -> None:
        super().__init__()
        self.name = name
        #self.father=None
        #self.attr= production/company
        self.decay = decay  #衰减系数
        self.r = r  #r，节点净库存流
        self.nbr = {}  #neighbor+关系 par/raw/st
        self.adj = []
        self.par = []
        self.pred = []
        self.parent=[]
        self.children=[]
    def update_r(self, change):
        self.r = round(self.r + change, 2)

    def get_item(self):
        return self.name
    def add_parent(self,name,re,i=0):
        self.parent.append(name)
    def add_child(self,name,re,i=0):
        self.children.append(name)
    def add_adj(self, name, re, i=0):
        self.nbr[name] = {'re': re, 'i': i}
        self.adj.append(name)

    def add_parity(self, name, re, i=0):
        self.nbr[name] = {'re': re, 'i': i}
        self.par.append(name)

    def add_pred(self, name, re, i=0):
        self.nbr[name] = {'re': re, 'i': i}
        self.pred.append(name)

    def __dict__(self):
        return {self.name: {x for x in self.nbr}}

    def get_r(self):
        return self.r

    def get_edge_list(self):
        # ('B', 'A', {'re': 'raw','i': 0}),
        edge_list = []
        for neighbor in self.nbr:
            edge_list.append((self.name, neighbor, self.nbr[neighbor]))
        return edge_list


class QGGraph():
    def __init__(self, ):
        self.node_dict = {}
        self.edge_dict = {}#还包含更新过程中继承自父节点、子节点的边
        self.edge_dict_const={}#只包含通过addedge添加的边
        self.edge_dict_temp={}#只包含从父节点、子节点继承的临时边
        self.updated_node_list = []
        self.node_i_out_dict = {}
        self.updated_edge_list = []
        self.num_nodes = 0
        self.num_edges = 0
        self.c_par = -0.5
        self.c_node = 0.8
        self.source_node = ''
        self.source_input = 0

    #构建和检索部分
    def add_Node(self, name):
        new_Node = QGNode(name)
        self.node_dict[name] = new_Node
        self.num_nodes += 1
        return new_Node

    def get_Node(self, name) -> dict:
        if name in self.node_dict:
            return self.node_dict[name]
        else:
            return None

    def get_all_Nodes(self) -> dict:
        dic = {}
        for node in self.node_dict:
            dic[node] = self.node_dict[node].get_r()
        return dic

    def add_Edge(self, f, t, re, i):
        if f not in self.node_dict:
            nn = self.add_Node(f)
        if t not in self.node_dict:
            nn = self.add_Node(t)
        if re in ('raw', 'st'):
            self.node_dict[f].add_adj(t, re, i)
            self.node_dict[t].add_pred(f, re, i)
        elif re == 'par':
            self.node_dict[f].add_parity(t, re, i)
            self.node_dict[t].add_parity(f, re, i)
        elif re=='parent':# 指t是f的父节点
            self.node_dict[f].add_parent(t,re,i)
            self.node_dict[t].add_child(f,re,i)
        elif re =='child':
            self.node_dict[f].add_child(t,re,i)
            self.node_dict[t].add_parent(f,re,i)
        else:
            raise QGGraphError('该关系不存在:',re)
        self.edge_dict[(f, t)] = {
            're': re,
            'i': i
        }  #在图里保存并更新所有边的信息,点只存变量r，不存边的i
        self.num_edges += 1
        self.edge_dict_const[(f, t)] = {
            're': re,
            'i': i
        }  
    def inherit(self):#将子节点、父节点的上下游进行继承,认定无多层父、子节点
        #eg node = r child=a
        for  name,node in self.node_dict.items():
            if node.children !=[]:
                for child in node.children:
                    for pred in self.node_dict[child].pred:
                        if pred not in node.pred:
                            node.pred+=pred
                            #self.edge_dict[(pred,name)]=self.edge_dict[(pred,child)]
                            self.edge_dict_temp[(pred,name)]=self.edge_dict[(pred,child)]
                    for adj in self.node_dict[child].adj:
                        if adj not in node.adj:
                            node.adj+=adj
                            self.edge_dict_temp[(name,adj)]=self.edge_dict[(child,adj)]
            if node.parent != []:
                for parent in node.parent:
                    for pred in self.node_dict[parent].pred:
                        if pred not in node.pred:
                            node.pred+=pred
                            self.edge_dict_temp[(pred,name)]=self.edge_dict[(pred,parent)]
                    for adj in self.node_dict[parent].adj:
                        if adj not in node.adj:
                            node.adj+=adj
                            self.edge_dict_temp[(name,adj)]=self.edge_dict[(parent,adj)]
                

    def get_all_Edges(self):
        return self.edge_dict

    # networkx 可视化部分
    def build_nx_dic(self):
        nx_node_list = []
        for n in self.node_dict:
            dic = {'node': self.node_dict[n], 'r': self.node_dict[n].get_r()}
            nx_node_list.append((n, dic))
        nx_edge_list = []
        for e in self.edge_dict_const:
            f, t = e
            nx_edge_list.append((f, t, self.edge_dict[e]))
        return nx_node_list, nx_edge_list

    def build_networkx(self):
        node, edge = self.build_nx_dic()
        G = nx.DiGraph()
        G.add_nodes_from(node)
        G.add_edges_from(edge)
        return G

    def draw_networkx(self):
        G = self.build_networkx()

        options = {
            "font_size": 15,
            "node_size": 1100,
            "node_color": "w",
            "edgecolors": "black",
            "linewidths": 0.5,
            "width": 1,
        }
        pos = nx.spring_layout(G, k=10000,
                               seed=6)  # Seed layout for reproducibility
        nx.draw(
            G,
            pos,
            **options,
        )
        node_labels = {}
        for node in G.nodes:
            node_labels[node] = str(node) + ': ' + str(G.nodes[node]['r'])
        nx.draw_networkx_labels(G, pos, labels=node_labels)
        edge_labels = {}
        for edge in G.edges:
            edge_labels[edge] = str(G[edge[0]][edge[1]]['re']) + ' ' + str(
                G[edge[0]][edge[1]]['i'])
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
        plt.show()

    # 计算、更新部分
    #轮子
    def update_Edge(self, f, t, i):
        if (f, t) in self.edge_dict:
            assert (f, t) not in self.updated_edge_list, ('该边已被更新过:', f, t)
            self.edge_dict[f, t]['i'] = round(i + self.edge_dict[f, t]['i'], 2)
            self.updated_edge_list.append((f, t))
        elif (t, f) in self.edge_dict:
            assert (t, f) not in self.updated_edge_list, ('该边已被更新过:', t, f)
            self.edge_dict[t, f]['i'] = round(i + self.edge_dict[t, f]['i'], 2)
            self.updated_edge_list.append((t, f))
        elif (f,t) in self.edge_dict_temp:
            assert (f, t) not in self.updated_edge_list, ('该边已被更新过:', f, t)
            self.edge_dict[(f, t)] = {'i':round(i , 2)}
            self.updated_edge_list.append((f, t))
        elif(t,f) in self.edge_dict_temp:
            assert (t, f) not in self.updated_edge_list, ('该边已被更新过:', t, f)
            self.edge_dict[(t, f)] = {'i':round(i , 2)}
            self.updated_edge_list.append((t, f))            
        else:
            #self.updated_edge_list.append((f,t))

            raise QGGraphError('更新的边不在图中存在:', f, t)

    def update_Node(self, name, r):
        assert name in self.node_dict, ('该节点不在图中:', name)
        assert name not in self.updated_node_list, ('该节点已被更新:', name)
        self.node_dict[name].update_r(r)
        self.updated_node_list.append(name)

    def get_pred_par(self, name):

        #假定互为同位的节点被全部输入，即两两间都为同位关系,获得分组列表
        query = []
        result = []
        for pred in self.node_dict[name].pred:
            group = [pred]
            if pred not in query:
                for par in self.node_dict[pred].par:
                    if par not in query:
                        if name in self.node_dict[par].adj:
                            group.append(par)
                            query.append(par)
                result.append(group)
                query.append(pred)
        return result
        
    def get_adj_par(self, name):
        query = []
        result = []
        for adj in self.node_dict[name].adj:
            group = [adj]
            if adj not in query:
                for par in self.node_dict[adj].par:
                    if par not in query:
                        if name in self.node_dict[par].pred:
                            group.append(par)
                            query.append(par)
                result.append(group)
                query.append(adj)
        return result
    #输入和响应
    def update_graph(self, input):
        self.add_source(input)
        self.source_node = input['f']
        self.source_input = input['i']
        q = [input['t']]
        while len(q) > 0:
            current = q.pop(0)
            nbr = []
            #print(current)
            #print(q)
            for adj in self.node_dict[current].adj:
                if adj not in self.updated_node_list:
                    q.append(adj)
                    #print(current,adj,self.edge_dict[current,adj]['i'])
                    self.update(f=current,
                                t=adj,
                                i_in=self.edge_dict[current, adj]['i'])
                for pred in self.node_dict[current].pred:
                    if pred not in self.updated_node_list:
                        q.append(pred)
                        #print(pred,current,self.edge_dict[pred,current]['i'])
                        self.update(f=current,
                                    t=pred,
                                    i_in=self.edge_dict[pred, current]['i'])
                '''for n in nbr:
                    q.append(n)
                    self.update(f=current,t=n,i_in=self.edge_dict[(current,n)])'''
        print('本次输入更新的边为:', self.updated_edge_list)
        print('本次输入更新的点为:', self.updated_node_list)
        self.updated_node_list = []
        self.updated_edge_list = []

    def add_source(self, input):
        #input={'f':'B','t':'A','i':20}
        f = input['f']
        t = input['t']
        i = input['i']
        self.update(f=f, t=t, i_in=i)

    def bfs(self, start):
        q = [start]
        over = []
        #start,i_out=
        while len(q) > 0:
            #if q[0] not in over:

            current = q.pop(0)

            print(current)
            nbr = []
            for adj in self.node_dict[current].adj:
                if adj not in self.updated_node_list:
                    nbr.append(adj)
            for pred in self.node_dict[current].pred:
                if pred not in self.updated_node_list:
                    nbr.append(pred)
            for n in nbr:
                print(n)
                q.append(n, )

                self.updated_node_list.append(n)
                print(self.updated_node_list)
            over.append(current)

    def update(self, f, t, i_in) -> None:

        #先分组，再计算，再递归传导
        #对一个点产生影响的只有一个源（或者经过同位衰减的一组源）
        #f='b',t='a',i_in=20
        #此处f，t为影响力传播，非边的方向,i为ft之间的影响力，同时为t的i_in更新前的初值
        #只对t更新，然后由t传播(与边的方向无关)
        if t == self.source_node:
            i_in = self.source_input
            self.update_Edge(f,t,i_in)
        c_node = self.c_node
        c_par = self.c_par
        preds = self.get_pred_par(t)
        adjs = self.get_adj_par(t)
        #print(t,adjs)
        f_node = self.node_dict[f]
        t_node = self.node_dict[t]
        if f in t_node.pred:
            #f为t的上游时，f-t是源，找上游组
            for group in preds:
                if f in group:
                    if len(group) > 1:
                        group.remove(f)
                        for other in group:
                            if (other, t) not in self.updated_edge_list:
                                self.update_Edge(
                                    other, t,
                                    round(c_par * i_in / len(group), 2))
                                i_in += round(c_par * i_in / len(group), 2)
                            else:

                                i_in += round(self.edge_dict[(other, t)]['i'],
                                              2)

        else:
            #否则相反，找下游组

            for group in adjs:
                
                if f in group:
                    #print(group)
                    if len(group) > 1:

                        group.remove(f)
                        for other in group:
                            if (t, other) not in self.updated_edge_list:
                                self.update_Edge(
                                    t, other,
                                    round(c_par * i_in / len(group), 2))
                                i_in += round(c_par * i_in / len(group), 2)
                                #print('trigger', i_in)
                            else:
                                i_in += round(self.edge_dict[(t, other)]['i'],
                                              2)
                                #print('trigger', i_in)
        #print(t, i_in)
        #更新ir
        if t not in self.updated_node_list:
            #print(t,round(i_in*(1-c_node),2))
            self.update_Node(t, r=round(i_in * (1 - c_node), 2))
        else:
            pass
        #对其他所有组更新iout
        i_out = round(i_in * c_node, 2)
        self.node_i_out_dict[t] = i_out
        for group in preds:
            num = len(group)
            for other in group:
                if (other, t) not in self.updated_edge_list:
                    self.update_Edge(other, t, round(i_out / num, 2))
        for group in adjs:
            num = len(group)
            for other in group:
                if (t, other) not in self.updated_edge_list:
                    self.update_Edge(t, other, round(i_out / num, 2))
        #更新相邻节点 #BFS
        '''others=self.node_dict[t].adj+self.node_dict[t].pred

        for pred in self.node_dict[t].pred:
            if pred not in self.updated_node_list:
                self.recursive_update(t,pred,i_out)
        for adj in self.node_dict[t].adj:
            if adj not in self.updated_node_list:
                self.recursive_update(t,adj,i_out)
        
        #更新i_in,如果有上游同位则同位衰减
        for par in self.node_dict[start].par:
            for adj in self.node_dict[par].adj:
                if adj in g.node_dict[start].adj:
                    if (par, adj) not in self.updated_edge_list:
                        #print(par)
                        self.update_Edge(par, adj, i=i_out * c_par)
            for pred in self.node_dict[par].pred:
                if pred in g.node_dict['B'].pred:
                    if(pred,par) not in self.updated_edge_list:
                        self.update_Edge(pred,par,i=i_in*c_par)
        i_out = i_in * c_node
        #更新其他边，如果有同位则分组
        '''

    '''def recursive_update(self, start):
        #start='B' start=f
        c_node = self.c_node
        c_par = self.c_par
        i_in=0
        #print('更新节点',start)
        #更新节点流经影响力
        for edge in self.edge_dict:
            if start in edge:
                i_in+=self.edge_dict[edge]['i']
        #print(start,i_in)
        i_out=round(i_in*c_node,2)
        #print(start,i_out)
        #更新节点所受影响力
        if start not in self.updated_node_list:
            if i_in !=0:  #此处写法存疑，会不会无限递归，是否有需要（是否真的存在例子中先F后K导致无强度）
                self.update_Node(start, i_in * (1 - c_node))
            else:
                return None
        #更新节点其他各边，出现同位时平分影响力
        for pred in self.node_dict[start].pred:
            divided_par=[pred]
            for par in self.node_dict[pred].par:
                if par in self.node_dict[start].pred:
                    divided_par.append(par)
            #divided_par 中存储需要平分影响力的边，如果没有同位则长度为1
            for pred in divided_par:
                #print(pred)
                if (pred,start) not in self.updated_edge_list:
                    self.update_Edge(pred,start,i=round(i_out/len(divided_par),2))
        for adj in self.node_dict[start].adj:
            divided_par=[adj]
            for par in self.node_dict[adj].par:
                if par in self.node_dict[start].adj:
                    divided_par.append(par)
            #divided_par 中存储需要平分影响力的边，如果没有同位则长度为1
            for adj in divided_par:
                #print(adj)
                if (start,adj) not in self.updated_edge_list:
                    self.update_Edge(start,adj,i=round(i_out/len(divided_par),2))
        #更新同位，且有共同pre/adj的边
        for par in self.node_dict[start].par:
            for adj in self.node_dict[par].adj:
                if adj in g.node_dict[start].adj:
                    if (par,adj) not in self.updated_edge_list:
                        #print(par)
                        self.update_Edge(par,adj,i=i_out*c_par)
            for pred in self.node_dict[par].pred:
                if pred in g.node_dict['B'].pred:
                    if(pred,par) not in self.updated_edge_list:
                        self.update_Edge(pred,par,i=i_in*c_par)
        #先更新抑制型同位
        for nbr in self.node_dict[start].par:
            if nbr not in self.updated_node_list:
                self.recursive_update(start=nbr)
        for nbr in self.node_dict[start].nbr:
            if nbr not in self.updated_node_list:
                self.recursive_update(start=nbr)'''


In [4]:
#g.node_dict
for k,v in g.node_dict.items():
    print(k,v.pred)


NameError: name 'g' is not defined

input:BA=20
    pred=B
    
    update(B):
        B=4
        B.i=16
        search B.pred :
            if has pr:
                B to prpreds=8
            else:
                B.preds=16
        search B.adj:
            if has adj:
                B to pradjs=8
            else:
                B.adjs=16
        if B.pr.adj in B.adj:
            CA=-10
        # updated all edges have B(ba bq bn  bo  and CA.....)
        ###### 一个源对所有节点和边只进行一次更新!如果已经更新则不再计算。edge的i直接赋值，node的r用老+新#######
        if all neighbors already updated:
            return QGgraph
        update(B.neighbors)
        
        A=2
        A.i=8
        AE=AF=4
        AD==AG=AJ=AI=8
        HI=-4
        update(A.neighbors)
        .......

In [24]:
nodes_list = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O',
    'P', 'Q','R','S'
]
g=QGGraph()
for n in nodes_list:
    g.add_Node(n)

In [25]:
whole_edges_list = [('B', 'A', {
    're': 'raw',
    'i': 0
}), ('C', 'A', {
    're': 'raw',
    'i': 0
}), ('D', 'A', {
    're': 'raw',
    'i': 0
}), ('E', 'A', {
    're': 'raw',
    'i': 0
}), ('F', 'A', {
    're': 'raw',
    'i': 0
}), ('G', 'A', {
    're': 'raw',
    'i': 0
}), ('K', 'G', {
    're': 'raw',
    'i': 0
}), ('A', 'I', {
    're': 'raw',
    'i': 0
}), ('A', 'J', {
    're': 'st',
    'i': 0
}), ('H', 'I', {
    're': 'raw',
    'i': 0
}), ('I', 'L', {
    're': 'raw',
    'i': 0
}), ('L', 'M', {
    're': 'raw',
    'i': 0
}), ('B', 'C', {
    're': 'par',
    'i': 0
}), ('A', 'H', {
    're': 'par',
    'i': 0
}), ('E', 'F', {
    're': 'par',
    'i': 0
}), ('F', 'K', {
    're': 'par',
    'i': 0
}), ('P', 'C', {
    're': 'raw',
    'i': 0
}), ('B', 'N', {
    're': 'raw',
    'i': 0
}), ('O', 'B', {
    're': 'raw',
    'i': 0
}), ('Q', 'B', {
    're': 'raw',
    'i': 0
}), ('Q', 'C', {
    're': 'raw',
    'i': 0
}), ('A', 'R', {
    're': 'parent',
    'i': 0
}), ('S', 'R', {
    're': 'raw',
    'i': 0
})]
for l in whole_edges_list:
    g.add_Edge(f=l[0], t=l[1], re=l[2]['re'], i=l[2]['i'])


In [28]:
g.edge_dict_const

{('B', 'A'): {'re': 'raw', 'i': 0},
 ('C', 'A'): {'re': 'raw', 'i': 0},
 ('D', 'A'): {'re': 'raw', 'i': 0},
 ('E', 'A'): {'re': 'raw', 'i': 0},
 ('F', 'A'): {'re': 'raw', 'i': 0},
 ('G', 'A'): {'re': 'raw', 'i': 0},
 ('K', 'G'): {'re': 'raw', 'i': 0},
 ('A', 'I'): {'re': 'raw', 'i': 0},
 ('A', 'J'): {'re': 'st', 'i': 0},
 ('H', 'I'): {'re': 'raw', 'i': 0},
 ('I', 'L'): {'re': 'raw', 'i': 0},
 ('L', 'M'): {'re': 'raw', 'i': 0},
 ('B', 'C'): {'re': 'par', 'i': 0},
 ('A', 'H'): {'re': 'par', 'i': 0},
 ('E', 'F'): {'re': 'par', 'i': 0},
 ('F', 'K'): {'re': 'par', 'i': 0},
 ('P', 'C'): {'re': 'raw', 'i': 0},
 ('B', 'N'): {'re': 'raw', 'i': 0},
 ('O', 'B'): {'re': 'raw', 'i': 0},
 ('Q', 'B'): {'re': 'raw', 'i': 0},
 ('Q', 'C'): {'re': 'raw', 'i': 0},
 ('A', 'R'): {'re': 'parent', 'i': 0},
 ('S', 'R'): {'re': 'raw', 'i': 0}}

In [26]:
g.draw_networkx()
print(g.get_all_Nodes())
g.get_all_Edges()


NameError: name 'nx' is not defined

In [21]:
g.update_graph(input={'f': 'B', 't': 'A', 'i': 20})
#g.add_source(input={'f': 'B', 't': 'A', 'i': 20})

本次输入更新的边为: [('C', 'A'), ('D', 'A'), ('E', 'A'), ('F', 'A'), ('G', 'A'), ('A', 'I'), ('A', 'J'), ('H', 'I'), ('I', 'L'), ('B', 'A'), ('O', 'B'), ('Q', 'B'), ('B', 'N'), ('P', 'C'), ('Q', 'C'), ('K', 'G'), ('L', 'M')]
本次输入更新的点为: ['A', 'I', 'B', 'C', 'D', 'E', 'F', 'G', 'J', 'L', 'H', 'O', 'Q', 'N', 'P', 'K', 'M']


In [22]:
g.draw_networkx()
print(g.get_all_Nodes())
g.get_all_Edges()

NameError: name 'nx' is not defined

In [270]:
new_list=[]
for node in nodes_list:
    locals()[node]=Node(node)
    new_list.append(eval(node))
new_list

NameError: name 'Node' is not defined

In [ ]:
import networkx as nx
G = nx.DiGraph()

In [ ]:
G = nx.DiGraph()


In [ ]:
nodes_list = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O',
    'P', 'Q'
]
new_list=[]
nodes_dict = {}
whole_nodes_list = []
for node in nodes_list:
    nodes_dict[node] = {'name': node, 'r': 0}
    whole_nodes_list.append((node, nodes_dict[node]))

whole_nodes_list


In [ ]:
G.add_nodes_from(whole_nodes_list)

In [ ]:
whole_edges_list

In [ ]:
G.add_edges_from(whole_edges_list)

In [ ]:
options = {
    "font_size": 15,
    "node_size": 1100,
    "node_color": "w",
    "edgecolors": "black",
    "linewidths": 0.5,
    "width": 1,

}
pos = nx.spring_layout(G, k=10000, seed=6)  # Seed layout for reproducibility
nx.draw(G, pos, **options,)
node_labels = {}
for node in G.nodes:
    node_labels[node] = str(G.nodes[node]['name'])+': '+str(G.nodes[node]['r'])
nx.draw_networkx_labels(G, pos, labels=node_labels)
edge_labels = {}
for edge in G.edges:
    edge_labels[edge] = str(G[edge[0]][edge[1]]['re'])+' '+ str(G[edge[0]][edge[1]]['i'])
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
plt.show()


In [ ]:
for pred,attr in G.pred['O'].items():
    print(pred)
    print(attr)

In [ ]:
# COMPUTE
def input(source=tuple):
    #输入
    c_par=-0.5
    c_node=0.8
    backward,forward,i=source
    G[backward][forward]['i']=i
    #对于流向相同节点的同位，设定相反衰减源
    for adj,attr in G.adj[backward].items(): # is well if node has no adj/pred
        if attr['re']=='par':
            if forward in G.adj[adj]:
                G[adj][forward]['i']=i*c_par
    for pred,attr in G.pred[backward].items():
        if attr['re']=='par':
            if forward in G.adj[pred]:
                G[pred][forward]['i']=i*c_par            
    return G    

In [ ]:
def update(node,g=G):
    #递归传导影响力
    #每个节点只更新一次
    #先由source更新总源
    #计算节点r和流经节点I
    ####和本节点同位的关系怎么算？
    ####直接上下游出现同位怎么算？
    c_pr=-0.5
    c_node=0.8
    




    pass    

In [ ]:
source=('B','A',20)
G=input(source)

In [ ]:
nodes_df=get_node_info(G)
print(nodes_df)


In [ ]:
edges_df=get_edge_info(G)
print(edges_df)

graph 中保存有 节点 节点所受影响力 r
边 权重(未定义) 关系 relationship=raw material=raw/ storage &transportation =st/parity 变影响力 i


In [ ]:
def get_node_info(g=G):
    nodes_df=pd.DataFrame(columns=('name','r'))
    nodes_df['name'],nodes_df['r']=zip(*list(G.nodes('r')))
    return nodes_df
def get_edge_info(g=G):
    edges_df=pd.DataFrame(columns=('from','to','re','i'))
    _,_,i=zip(*list(G.edges(data='i')))
    edges_df['from'],edges_df['to'],edges_df['re']=zip(*list(G.edges(data='re')))
    edges_df['i']=i
    return edges_df
      

In [ ]:
nodes_df=get_node_info(G)
print(nodes_df)
edges_df=get_edge_info(G)
print(edges_df)

In [ ]:
list(G.edges())

In [ ]:
G.nodes['A']


In [ ]:
G.number_of_nodes()


In [ ]:
G.number_of_edges()

In [ ]:
G.adj['Q']

In [ ]:
'A' in G.pred['A']

In [ ]:
H = nx.Graph(G)
nx.draw(H, pos, **options)


In [ ]:
H.edges

In [ ]:
I=nx.DiGraph(H)
nx.draw(I,pos,**options)